In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_grouped_normalized_attention(
    per_tag_csv,
    global_csv,
    tag_order=None,
    tag_rename=None,
    colors=None,
    chunk_to_stack=None,
    xlabel="",
    ylabel="Mean attention − global mean",
    title="",
    figsize=(6.5, 3.5),
    bar_width=0.18,
    linewidth=0.6,
    tick_fontsize=13,
    label_fontsize=12,
    title_fontsize=10,
    plot_std=False,
    show_spines=False,
    legend_loc="upper right",
    legend_fontsize=12,
    legend_ncol=2,
    legend_frame=False,
    show_top_xticks=False,
    show_trend_lines=False,
    merge_tags=None, 
):
    """
    Grouped bar plot with merged tags capability and centered, wrapped x-labels.
    """

    # ------------------
    # Default chunk mapping
    # ------------------
    if chunk_to_stack is None:
        chunk_to_stack = {
            "currently_generating_token__attends_to__image": "Image",
            "currently_generating_token__attends_to__text": "Text",
            "currently_generating_token__attends_to__instruction": "Instruction",
            "currently_generating_token__attends_to__previously_generating_tokens": "Previous",
        }

    STACKS = list(dict.fromkeys(chunk_to_stack.values()))

    # ------------------
    # Color validation
    # ------------------
    if colors is None:
        raise ValueError("You must provide `colors` explicitly.")

    missing_colors = set(STACKS) - set(colors)
    if missing_colors:
        raise ValueError(f"Missing colors for stacks: {missing_colors}")

    # ------------------
    # Load data
    # ------------------
    df = pd.read_csv(per_tag_csv)
    global_df = pd.read_csv(global_csv)

    df = df[df["chunk"].isin(chunk_to_stack)]
    df["stack"] = df["chunk"].map(chunk_to_stack)

    # ------------------
    # Pivot
    # ------------------
    mean_pivot = (
        df.pivot_table(index="tag", columns="stack", values="mean", aggfunc="mean")
        .reset_index()
    )

    std_pivot = (
        df.pivot_table(index="tag", columns="stack", values="std", aggfunc="mean")
        .reset_index()
    )

    # ------------------
    # Normalize against global means
    # ------------------
    global_mean = dict(zip(global_df["Metric"], global_df["Mean"]))

    for chunk, stack in chunk_to_stack.items():
        if chunk not in global_mean:
            raise ValueError(f"Missing global mean for chunk: {chunk}")
        mean_pivot[stack] -= global_mean[chunk]

    # ------------------
    # Merge Tags
    # ------------------
    if merge_tags:
        for keep_tag, remove_tag in merge_tags:
            if (keep_tag in mean_pivot['tag'].values) and (remove_tag in mean_pivot['tag'].values):
                # Update Means
                keep_vals = mean_pivot.loc[mean_pivot['tag'] == keep_tag, STACKS].values
                remove_vals = mean_pivot.loc[mean_pivot['tag'] == remove_tag, STACKS].values
                mean_pivot.loc[mean_pivot['tag'] == keep_tag, STACKS] = (keep_vals + remove_vals) / 2.0
                
                # Update Stds
                keep_std = std_pivot.loc[std_pivot['tag'] == keep_tag, STACKS].values
                remove_std = std_pivot.loc[std_pivot['tag'] == remove_tag, STACKS].values
                std_pivot.loc[std_pivot['tag'] == keep_tag, STACKS] = (keep_std + remove_std) / 2.0

                # Remove old tag
                mean_pivot = mean_pivot[mean_pivot['tag'] != remove_tag]
                std_pivot = std_pivot[std_pivot['tag'] != remove_tag]
            else:
                print(f"Warning: Could not merge {remove_tag} into {keep_tag}.")

    # ------------------
    # Ordering
    # ------------------
    if tag_order is not None:
        available_tags = set(mean_pivot["tag"].unique())
        filtered_order = [t for t in tag_order if t in available_tags]
        
        mean_pivot["tag"] = pd.Categorical(mean_pivot["tag"], filtered_order, ordered=True)
        std_pivot["tag"] = pd.Categorical(std_pivot["tag"], filtered_order, ordered=True)
        mean_pivot = mean_pivot.sort_values("tag")
        std_pivot = std_pivot.sort_values("tag")
        
        mean_pivot = mean_pivot.dropna(subset=["tag"])
        std_pivot = std_pivot.dropna(subset=["tag"])

    # ------------------
    # Plot
    # ------------------
    x = np.arange(len(mean_pivot))
    fig, ax = plt.subplots(figsize=figsize)

    for i, stack in enumerate(STACKS):
        y = mean_pivot[stack].values
        yerr = std_pivot[stack].values if plot_std else None

        ax.bar(
            x + i * bar_width,
            y,
            bar_width,
            color=colors[stack],
            linewidth=linewidth,
            label=stack,
            yerr=yerr,
            error_kw=dict(
                elinewidth=0.6,
                capsize=1.5,
                capthick=0.6,
            ) if plot_std else None,
        )
    
    if show_trend_lines:
        for i, stack in enumerate(STACKS):
            y = mean_pivot[stack].values
            x_centers = x + i * bar_width
            ax.plot(x_centers, y, color=colors[stack], linewidth=0.1, alpha=0.9, zorder=3)

    # ------------------
    # Axes formatting (UPDATED)
    # ------------------
    ax.axhline(0, linewidth=0.6)

    raw_xticks = mean_pivot["tag"].astype(str).tolist()
    
    # 1. Rename tags using dict
    # 2. Wrap text: replace spaces with newlines
    final_xticks = []
    for t in raw_xticks:
        label = tag_rename.get(t, t) if tag_rename else t
        # Force wrap on space
        label = label.replace(" ", "\n")
        final_xticks.append(label)

    # Center position of the group
    xtick_pos = x + bar_width * (len(STACKS) - 1) / 2

    ax.set_xticks(
        xtick_pos,
        final_xticks,
        rotation=0,       # No rotation
        ha="center",      # Center alignment
        fontsize=tick_fontsize,
    )
    ax.tick_params(axis="y", labelsize=tick_fontsize)

    ax.set_xlabel(xlabel, fontsize=label_fontsize)
    ax.set_ylabel(ylabel, fontsize=label_fontsize)
    ax.set_title(title, fontsize=title_fontsize)

    # ------------------
    # Optional top x-axis
    # ------------------
    ax_top = None
    if show_top_xticks:
        ax_top = ax.twiny()
        ax_top.set_xlim(ax.get_xlim())
        ax_top.set_xticks(xtick_pos)
        ax_top.set_xticklabels(final_xticks, rotation=0, ha="center", fontsize=tick_fontsize)
        ax_top.tick_params(axis="x", length=0)

    # ------------------
    # Legend
    # ------------------
    ax.legend(
        loc=legend_loc,
        fontsize=legend_fontsize,
        ncol=legend_ncol,
        frameon=legend_frame,
        handlelength=1.2,
        handletextpad=0.4,
        columnspacing=0.8,
    )

    # ------------------
    # Spines
    # ------------------
    if not show_spines:
        for spine in ["top", "right", "left", "bottom"]:
            ax.spines[spine].set_visible(False)
            if ax_top is not None:
                ax_top.spines[spine].set_visible(False)

        ax.tick_params(axis="x", length=0)
        ax.tick_params(axis="y", length=0)
        if ax_top is not None:
            ax_top.tick_params(axis="x", length=0)

    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
colors = {
    "Image": "#FFA500",
    "Text": "#267E59",
    "Instruction": "#D25B5B",
    "Previous": "#C01BA7",
}


tag_rename = {
    "SOG": "SOG",
    "Q1_INTRO": "Q1 Intro",
    "Q1_ANSWER": "Q1 Answer",
    "FIRST_HANDOFF": "Handoff",
    "OTHER_HANDOFF": "Handoff",
    "Q2_INTRO": "Q2 Intro",
    "Q2_ANSWER": "Q2 Answer",
    "POSTAMBLE": "EOG",
}
tag_order = [
        "SOG", "Q1_INTRO", "Q1_ANSWER",
        "FIRST_HANDOFF", "OTHER_HANDOFF",
        "Q2_INTRO", "Q2_ANSWER", "POSTAMBLE",
    ]


fig = plot_grouped_normalized_attention(
    per_tag_csv="<RUN_ROOT>/otat/ChartQA/gemma4_12b/charta_gemma4_attention_compiled.csv",
    global_csv="<RUN_ROOT>/otat/ChartQA/gemma4_12b/global_means/ChartQA__gemma4_12b.csv",
    tag_order=tag_order,
    tag_rename=tag_rename,
    # xlabel="Generation stage",
    ylabel="Normalized Attention",
    # title="  Fruit-Math: Qwen2.5VL-3B",
    legend_loc="upper left",
    # legend_fontsize=10,
    legend_ncol=4,
    plot_std=True,
    show_spines=False,
    colors=colors,
    merge_tags=[("FIRST_HANDOFF", "OTHER_HANDOFF")],
    
)

#save hires figure
fig.savefig("<REPO_ROOT>/data/plots/bar_plots/ChartQA_Final/gemma4_12b.pdf", dpi=300)


In [ ]:
fig = plot_grouped_normalized_attention(
    per_tag_csv="<REPO_ROOT>/data/bar_plot_csv/chartQA/lov_7b.csv",
    global_csv="<REPO_ROOT>/data/global_means/chartQA/ChartQA__lov_7b.csv",
    tag_order=tag_order,
    tag_rename=tag_rename,
    # xlabel="Generation stage",
    ylabel="Normalized Attention",
    # title="  Fruit-Math: Qwen2.5VL-3B",
    legend_loc="upper left",
    # legend_fontsize=10,
    legend_ncol=4,
    plot_std=True,
    show_spines=False,
    colors=colors,
    merge_tags=[("FIRST_HANDOFF", "OTHER_HANDOFF")],
    
)

#save hires figure
fig.savefig("<REPO_ROOT>/data/plots/bar_plots/ChartQA_Final/lov_7b.pdf", dpi=300)


In [ ]:
fig = plot_grouped_normalized_attention(
    per_tag_csv="<REPO_ROOT>/data/bar_plot_csv/chartQA/qvl_3b.csv",
    global_csv="<REPO_ROOT>/data/global_means/chartQA/ChartQA__qvl_3b.csv",
    tag_order=tag_order,
    tag_rename=tag_rename,
    # xlabel="Generation stage",
    ylabel="Normalized Attention",
    # title="  Fruit-Math: Qwen2.5VL-3B",
    legend_loc="upper left",
    # legend_fontsize=10,
    legend_ncol=4,
    plot_std=True,
    show_spines=False,
    colors=colors,
    merge_tags=[("FIRST_HANDOFF", "OTHER_HANDOFF")],
    
)

#save hires figure
fig.savefig("<REPO_ROOT>/data/plots/bar_plots/ChartQA_Final/qvl_3b.pdf", dpi=300)


In [ ]:
fig = plot_grouped_normalized_attention(
    per_tag_csv="<RUN_ROOT>/otat/ChartQA/qvl_7b/charta_qvl7b_attention_compiled.csv",
    global_csv="<RUN_ROOT>/otat/ChartQA/qvl_7b/global_means/ChartQA__qvl_7b.csv",
    tag_order=tag_order,
    tag_rename=tag_rename,
    # xlabel="Generation stage",
    ylabel="Normalized Attention",
    # title="  Fruit-Math: Qwen2.5VL-3B",
    legend_loc="upper left",
    # legend_fontsize=10,
    legend_ncol=4,
    plot_std=True,
    show_spines=False,
    colors=colors,
    merge_tags=[("FIRST_HANDOFF", "OTHER_HANDOFF")],
    
)

#save hires figure
fig.savefig("<REPO_ROOT>/data/plots/bar_plots/ChartQA_Final/qvl_7b.pdf", dpi=300)


# Layer Grouped Plotting

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_layer_group_attention(
    per_tag_csv,
    tag_order=None,
    tag_rename=None,
    colors=None,
    chunk_to_stack=None,
    layer_groups=("early", "mid", "late"),
    group_styles=None,
    normalize=True,           # subtract per-chunk-group global mean from the CSV itself
    xlabel="",
    ylabel="Mean attention − global mean",
    title="",
    figsize=(8, 3.5),
    bar_width=0.08,
    group_gap=0.04,
    linewidth=0.6,
    tick_fontsize=13,
    label_fontsize=12,
    title_fontsize=10,
    plot_std=False,
    show_spines=False,
    legend_loc="upper right",
    legend_fontsize=10,
    legend_ncol=2,
    legend_frame=False,
    merge_tags=None,
    show_table=False
):
    if chunk_to_stack is None:
        chunk_to_stack = {
            "currently_generating_token__attends_to__image":                        "Image",
            "currently_generating_token__attends_to__text":                         "Text",
            "currently_generating_token__attends_to__instruction":                  "Instruction",
            "currently_generating_token__attends_to__previously_generating_tokens": "Previous",
        }

    STACKS = list(dict.fromkeys(chunk_to_stack.values()))
    GROUPS = list(layer_groups)

    if colors is None:
        raise ValueError("Provide `colors` dict keyed by stack name.")
    missing = set(STACKS) - set(colors)
    if missing:
        raise ValueError(f"Missing colors for stacks: {missing}")

    if group_styles is None:
        group_styles = {
            "early": dict(hatch="",    alpha=1.00),
            "mid":   dict(hatch="//",  alpha=0.85),
            "late":  dict(hatch="xx",  alpha=0.70),
        }

    # ── load & filter ─────────────────────────────────────────────────────────
    df = pd.read_csv(per_tag_csv)
    df = df[df["chunk"].isin(chunk_to_stack) & df["layer_group"].isin(GROUPS)].copy()
    df["stack"] = df["chunk"].map(chunk_to_stack)

    # ── normalize: subtract mean-of-means per (chunk, layer_group) ────────────
    if normalize:
        global_means = (
            df.groupby(["chunk", "layer_group"])["mean"]
            .mean()
            .rename("global_mean")
            .reset_index()
        )
        df = df.merge(global_means, on=["chunk", "layer_group"])
        df["mean"] = df["mean"] - df["global_mean"]

    # ── pivot ─────────────────────────────────────────────────────────────────
    mean_pivot = df.pivot_table(
        index="tag", columns=["stack", "layer_group"], values="mean", aggfunc="mean"
    )
    std_pivot = df.pivot_table(
        index="tag", columns=["stack", "layer_group"], values="std", aggfunc="mean"
    )

    mean_pivot = mean_pivot.reset_index()
    std_pivot  = std_pivot.reset_index()

    # ── fix MultiIndex columns after reset_index ───────────────────────────────
    mean_pivot.columns = [
        c[0] if c[1] == "" else c          # ("tag", "") → "tag"; ("Image","early") stays a tuple
        for c in mean_pivot.columns
    ]
    std_pivot.columns = [
        c[0] if c[1] == "" else c
        for c in std_pivot.columns
    ]

    # ── merge tags ────────────────────────────────────────────────────────────
    if merge_tags:
        cols = [(s, g) for s in STACKS for g in GROUPS]
        # keep only cols that exist
        cols = [c for c in cols if c in mean_pivot.columns]
        for keep_tag, remove_tag in merge_tags:
            if keep_tag in mean_pivot["tag"].values and remove_tag in mean_pivot["tag"].values:
                k_m = mean_pivot.loc[mean_pivot["tag"] == keep_tag,  cols].values
                r_m = mean_pivot.loc[mean_pivot["tag"] == remove_tag, cols].values
                mean_pivot.loc[mean_pivot["tag"] == keep_tag, cols] = (k_m + r_m) / 2.0

                k_s = std_pivot.loc[std_pivot["tag"] == keep_tag,  cols].values
                r_s = std_pivot.loc[std_pivot["tag"] == remove_tag, cols].values
                std_pivot.loc[std_pivot["tag"] == keep_tag, cols] = (k_s + r_s) / 2.0

                mean_pivot = mean_pivot[mean_pivot["tag"] != remove_tag]
                std_pivot  = std_pivot[std_pivot["tag"]  != remove_tag]
            else:
                print(f"Warning: could not merge '{remove_tag}' into '{keep_tag}'.")

    # ── ordering ──────────────────────────────────────────────────────────────
    if tag_order is not None:
        available = set(mean_pivot["tag"].unique())
        filtered  = [t for t in tag_order if t in available]
        mean_pivot["tag"] = pd.Categorical(mean_pivot["tag"], filtered, ordered=True)
        std_pivot["tag"]  = pd.Categorical(std_pivot["tag"],  filtered, ordered=True)
        mean_pivot = mean_pivot.sort_values("tag").dropna(subset=["tag"])
        std_pivot  = std_pivot.sort_values("tag").dropna(subset=["tag"])

    n_tags   = len(mean_pivot)
    n_groups = len(GROUPS)
    cluster_width  = n_groups * bar_width
    total_per_tag  = len(STACKS) * cluster_width + (len(STACKS) - 1) * group_gap
    x = np.arange(n_tags) * (total_per_tag + 0.25)

    # ── plot ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)
    legend_handles = {}

    for si, stack in enumerate(STACKS):
        stack_offset = si * (cluster_width + group_gap)
        for gi, group in enumerate(GROUPS):
            col = (stack, group)
            if col not in mean_pivot.columns:
                continue

            y    = mean_pivot[col].values.astype(float)
            yerr = std_pivot[col].values.astype(float) if plot_std else None
            style = group_styles.get(group, {})

            ax.bar(
                x + stack_offset + gi * bar_width,
                y,
                bar_width,
                color=colors[stack],
                linewidth=linewidth,
                yerr=yerr,
                error_kw=dict(elinewidth=0.6, capsize=1.5, capthick=0.6) if plot_std else None,
                **style,
            )

            if stack not in legend_handles:
                legend_handles[stack] = plt.Rectangle(
                    (0, 0), 1, 1, fc=colors[stack], ec="none", label=stack,
                )
            if group not in legend_handles:
                legend_handles[group] = plt.Rectangle(
                    (0, 0), 1, 1,
                    fc="grey",
                    hatch=style.get("hatch", ""),
                    alpha=style.get("alpha", 1.0),
                    ec="white",
                    label=group.capitalize(),
                )

    # ── axes ──────────────────────────────────────────────────────────────────
    ax.axhline(0, linewidth=0.6, color="black")

    xtick_pos   = x + total_per_tag / 2 - bar_width / 2
    raw_labels  = mean_pivot["tag"].astype(str).tolist()
    final_labels = [
        (tag_rename.get(t, t) if tag_rename else t).replace(" ", "\n")
        for t in raw_labels
    ]
    ax.set_xticks(xtick_pos)
    ax.set_xticklabels(final_labels, rotation=0, ha="center", fontsize=tick_fontsize)
    ax.tick_params(axis="y", labelsize=tick_fontsize)
    ax.set_xlabel(xlabel, fontsize=label_fontsize)
    ax.set_ylabel(ylabel, fontsize=label_fontsize)
    ax.set_title(title, fontsize=title_fontsize)

    ordered_handles = (
        [legend_handles[s] for s in STACKS if s in legend_handles] +
        [legend_handles[g] for g in GROUPS  if g in legend_handles]
    )
    ax.legend(
        handles=ordered_handles,
        loc=legend_loc,
        fontsize=legend_fontsize,
        ncol=legend_ncol,
        frameon=legend_frame,
        handlelength=1.2,
        handletextpad=0.4,
        columnspacing=0.8,
    )

    if not show_spines:
        for spine in ["top", "right", "left", "bottom"]:
            ax.spines[spine].set_visible(False)
        ax.tick_params(axis="both", length=0)

    plt.tight_layout(pad=0.4)
    plt.show()

    if show_table:
        from tabulate import tabulate

# inside the function, add show_table=False to the signature, then at the end:

    if show_table:
        # Build a readable table: rows=tags, cols=(stack, group)
        rows = []
        for _, row in mean_pivot.iterrows():
            tag = str(row["tag"])
            entry = [tag]
            for stack in STACKS:
                for group in GROUPS:
                    col = (stack, group)
                    val = row[col] if col in mean_pivot.columns else float("nan")
                    entry.append(f"{val:+.4f}" if not np.isnan(val) else "—")
            rows.append(entry)

        headers = ["Tag"] + [f"{s}\n{g}" for s in STACKS for g in GROUPS]

        print(tabulate(
            rows,
            headers=headers,
            # tablefmt="rounded_outline",
            # colalign=["left"] + ["center"] * (len(STACKS) * len(GROUPS)),
        ))

    return fig

In [ ]:
plot_layer_group_attention(
    per_tag_csv="./data_plotting/bar_plot_csv_data_layerGrouped/vanilla/fruit_math/q25vl3B.csv",  # new layer-group CSV

    chunk_to_stack={
        "currently_generating_token__attends_to__image":                   "Image",
        "currently_generating_token__attends_to__text":                    "Text",
        "currently_generating_token__attends_to__instruction":                  "Instruction",
        "currently_generating_token__attends_to__previously_generating_tokens": "Previous",
    },

    tag_order = [
        "SOG", "FRUIT_INTRO", "FRUIT_CONCEPT",
        "FIRST_HANDOFF", "OTHER_HANDOFF",
        "MATH_INTRO", "MATH_ANSWER", "POSTAMBLE",
    ],
    colors = {
    "Image": "#FFA500",
    "Text": "#267E59",
    "Instruction": "#D25B5B",
    "Previous": "#C01BA7",
    },

    # Optional: override default hatch/alpha per layer group
    group_styles={
        "early": dict(hatch="",   alpha=1.00),
        "mid":   dict(hatch="//", alpha=0.85),
        "late":  dict(hatch="xx", alpha=0.70),
    },

    figsize=(18, 4),
    bar_width=0.07,
    group_gap=0.05,
    ylabel="Mean attention − global mean",
    merge_tags=[("FRUIT_CONCEPT", "FIRST_HANDOFF")],
    show_table=True,
);